In [1]:
import os
from blandai_caller import send_call, retrieve_and_save_transcripts
from transcript_analyzer import analyze_and_save_answers
import pandas as pd
import json
from utils import  get_codebook_object_from_json, convert_textdf_to_numeric_response_df
from dotenv import load_dotenv
from redcap_manager import upload_data_to_redcap

load_dotenv()
PHONE_NUMBER = os.getenv("PHONE_NUMBER")

In [2]:
#define participant id
participant_id = 17

In [3]:
#read task prompt for bland
with open("blandprompt.txt", "r") as f:
    task_prompt = f.read()

#get codebook file of covid impact survey
codebook_file = "./codebook/codebook.json"
with open(codebook_file, 'r') as file:
    codebook_json = json.load(file)
codebook = get_codebook_object_from_json(codebook_json)

In [4]:
#make a call to conduct survey
call_id = send_call(task_prompt, PHONE_NUMBER, conversational_model = "base")

In [5]:
#This step should be done after the call has ended.

#get a converstion transcript
if call_id is not None:
    call_ids = [call_id]
    transcript_file = retrieve_and_save_transcripts(participant_id, call_ids, blandai_data_dir = './blandai-data')
    #print(transcript_file)

In [6]:
#gpt-4o deduction of user responses, resulting file is a csv file
resulting_file = analyze_and_save_answers(participant_id, codebook_file, transcript_file)

In [7]:
#load the resulting file
df                    = pd.read_csv(resulting_file)
df.head()

,AGE7,GENDER,RACETH,HHINCOME,EDUCATION,HHSIZE1,HH01S,HH25S,HH612S,HH1317S,...,PHYS1C,PHYS1D,PHYS1E,PHYS1F,PHYS1G,PHYS1H,PHYS1I,PHYS1J,PHYS11,PHYS11_TEMP
0,25-34,Male,Korean,OTHER,Professional or Doctorate degree,"One person, I live by myself",0,0,0,0,...,No,Yes,Yes,No,No,No,No,No,Yes,97.7


In [8]:
#convert resulting files text responses to numeric responses
numeric_df            = convert_textdf_to_numeric_response_df(codebook, df)

#add participant id and survey information for redcap upload
numeric_df['record_id'] = pd.Series([participant_id+1])
numeric_df['conversational_agent_survey_complete'] = pd.Series(['2'])

numeric_df.head()

,AGE7,GENDER,RACETH,HHINCOME,EDUCATION,HHSIZE1,HH01S,HH25S,HH612S,HH1317S,...,PHYS1E,PHYS1F,PHYS1G,PHYS1H,PHYS1I,PHYS1J,PHYS11,PHYS11_TEMP,record_id,conversational_agent_survey_complete
0,2,1,8,404,14,1,0,0,0,0,...,1,2,2,2,2,2,1,97.7,18,2


In [9]:
#Upload data to redcap
upload_data_to_redcap(numeric_df)

REDCap upload was successful.
